# 02 – Streamflow Forecasting
**AquaML Portfolio Project**

Daily streamflow prediction using classical ML and deep learning.

**Models compared**
- Persistence baseline
- XGBoost (lag features)
- LSTM (sequence model)

**Metrics**: NSE, KGE, RMSE, MAE (standard in hydrology)


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.preprocessing import create_lag_features, create_sequences, temporal_train_test_split, scale_features
from src.evaluation import regression_report, nash_sutcliffe, kling_gupta
from src.visualization import plot_timeseries_forecast, plot_residuals

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Load & prepare data

In [ ]:
sf = pd.read_csv(ROOT / "data/sample/streamflow_daily.csv", parse_dates=["date"])
sf = create_lag_features(sf, "streamflow_m3s", lags=[1,2,3,7,14], extra_cols=["precip_mm", "temp_c"])
train_df, test_df = temporal_train_test_split(sf, test_ratio=0.2)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")


## Baseline: Persistence

In [ ]:
y_true = test_df["streamflow_m3s"].values
y_persist = test_df["streamflow_m3s_lag1"].values
print(regression_report(y_true, y_persist, prefix="Persist_"))


## XGBoost with lag features

In [ ]:
feature_cols = [c for c in train_df.columns if c not in ["date", "streamflow_m3s"]]
X_train, y_train = train_df[feature_cols].values, train_df["streamflow_m3s"].values
X_test, y_test = test_df[feature_cols].values, test_df["streamflow_m3s"].values

xgb = XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
xgb.fit(X_train, y_train)
y_xgb = xgb.predict(X_test)
print(regression_report(y_test, y_xgb, prefix="XGB_"))


## LSTM Sequence Model

In [ ]:
# Rebuild sequences from original (before lag features)
sf_raw = pd.read_csv(ROOT / "data/sample/streamflow_daily.csv", parse_dates=["date"])
feat_cols = ["streamflow_m3s", "precip_mm", "temp_c"]
data = sf_raw[feat_cols].values.astype(np.float32)
target = sf_raw["streamflow_m3s"].values.astype(np.float32)

seq_len, horizon = 30, 1
X, y = create_sequences(data, target, seq_len=seq_len, horizon=horizon)
y = y.ravel()

split = int(len(X) * 0.8)
X_tr, X_te = X[:split], X[split:]
y_tr, y_te = y[:split], y[split:]

# Scale using training statistics only
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
n_feat = X_tr.shape[-1]
X_tr_s = scaler.fit_transform(X_tr.reshape(-1, n_feat)).reshape(X_tr.shape)
X_te_s = scaler.transform(X_te.reshape(-1, n_feat)).reshape(X_te.shape)

class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

model = LSTMForecaster(n_feat).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_tr_s), torch.tensor(y_tr)),
    batch_size=64, shuffle=True
)

model.train()
for epoch in range(40):
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | loss {total/len(X_tr_s):.4f}")

model.eval()
with torch.no_grad():
    y_lstm = model(torch.tensor(X_te_s).to(device)).cpu().numpy()
print(regression_report(y_te, y_lstm, prefix="LSTM_"))


## Visual comparison (XGBoost)

In [ ]:
dates_te = test_df["date"].values
plot_timeseries_forecast(dates_te, y_test, y_xgb, title="XGBoost Streamflow Forecast")
plt.show()
plot_residuals(y_test, y_xgb, title="XGBoost Residuals")
plt.show()


## Takeaways
- Persistence is a strong baseline for short horizons.
- XGBoost with lag + meteorological features usually outperforms simple persistence.
- LSTM captures longer temporal dependencies; with more data and tuning it often wins on multi-day horizons.
- Always report **NSE** and **KGE** alongside RMSE for hydrological relevance.
